In [1]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_transformer
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    IsolationForest,
)
from sklearn.neighbors import KNeighborsRegressor
from scipy.stats import randint, uniform
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV


random_state = 42
np.random.seed(random_state)

In [2]:
pd.set_option('display.max_columns', None)

df = pd.read_csv('../../data/processed_player_data_v2.csv')

df.head()

,Game ID,player_rating,opponent_rating,player_white,opening_ply,elo_delta_ratio,opening_frequency,player_centipawn_loss,opponent_centipawn_loss,time_control,time_control_encoded,player_username,opponent_username,opening_familiarity,player_rapid_rating,opponent_rapid_rating,player_rapid_games,opponent_rapid_games,player_blitz_rating,opponent_blitz_rating,player_blitz_games,opponent_blitz_games,player_bullet_rating,opponent_bullet_rating,player_bullet_games,opponent_bullet_games,player_total_games,opponent_total_games,player_play_time_secs,opponent_play_time_secs,player_avg_game_secs,opponent_avg_game_secs,player_blitz_vs_rapid,player_bullet_vs_blitz,player_bullet_vs_rapid,opponent_blitz_vs_rapid,opponent_bullet_vs_blitz,opponent_bullet_vs_rapid,player_rapid_ratio,player_blitz_ratio,player_bullet_ratio,opponent_rapid_ratio,opponent_blitz_ratio,opponent_bullet_ratio,rapid_rating_gap,blitz_rating_gap,total_games_gap,rapid_games_gap,play_time_gap
0,J7Xvjkte,1441,1559,1,4,-0.078667,2,32,87,rapid,3,jerzypa2,timothei,1,1467.0,1659.0,965,2657,1155.0,1385.0,142,952,982.0,1018.0,26,8,1251,3980,1016708,3090613,812.716227,776.535930,-312.0,-173.0,-485.0,-274.0,-367.0,-641.0,0.771383,0.113509,0.020783,0.667588,0.239196,0.002010,-192.0,-230.0,-2729,-1692,-2073905
1,SSzpi7W1,1258,1567,1,6,-0.218761,2,43,6,rapid,3,nephi,veky,1,1221.0,1689.0,261,3466,1184.0,1393.0,6,9327,1284.0,1500.0,3,4452,333,18177,272224,6513529,817.489489,358.339055,-37.0,100.0,63.0,-296.0,107.0,-189.0,0.783784,0.018018,0.009009,0.190681,0.513121,0.244925,-468.0,-209.0,-17844,-3205,-6241305
2,NHpcYm3r,1697,1712,1,7,-0.008800,2,11,42,rapid,3,ronuh,kuramateca,1,NaN,1573.0,0,9230,NaN,1357.0,0,119,NaN,1330.0,0,2,0,9358,0,7732082,NaN,826.253687,NaN,NaN,NaN,-216.0,-27.0,-243.0,NaN,NaN,NaN,0.986322,0.012716,0.000214,NaN,NaN,-9358,-9230,-7732082
3,Nrzmgzmn,1978,1868,1,5,0.057202,2,21,53,rapid,3,anaya44,jeoda,1,NaN,1786.0,0,2172,NaN,1640.0,0,29436,NaN,955.0,0,1,0,31613,0,16836309,NaN,532.575491,NaN,NaN,NaN,-146.0,-685.0,-831.0,NaN,NaN,NaN,0.068706,0.931136,0.000032,NaN,NaN,-31613,-2172,-16836309
4,xMuFsnC6,2073,1816,1,2,0.132168,2,35,50,rapid,3,marc_robin,deraincharles,1,2158.0,2029.0,226,790,2119.0,1841.0,31948,2351,1500.0,1609.0,0,1357,32177,4567,14125375,1843793,438.989806,403.720823,-39.0,-619.0,-658.0,-188.0,-232.0,-420.0,0.007024,0.992883,0.000000,0.172980,0.514780,0.297132,129.0,278.0,27610,-564,12281582


In [3]:
df = df.drop(columns=['Game ID', 'player_username', 'opponent_username', 'time_control','time_control_encoded','opponent_centipawn_loss'])
rating_cols = ['player_rapid_rating', 'player_blitz_rating', 'player_bullet_rating']
opponent_rating_cols = ['opponent_rapid_rating', 'opponent_blitz_rating', 'opponent_bullet_rating']

df['player_profile_disabled']   = df[rating_cols].isnull().any(axis=1).astype(int)
df['opponent_profile_disabled']  = df[opponent_rating_cols].isnull().any(axis=1).astype(int)

df_original = df.copy()

In [4]:
X = df.drop(columns=['player_centipawn_loss'])
y = df['player_centipawn_loss']

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42)

Best params from Randomized Search: {'model__learning_rate': np.float64(0.027425083650459835), 'model__max_depth': 7, 'model__max_iter': 199}
CV RMSE:     34.19

In [ ]:

kf = KFold(n_splits=5, shuffle=True, random_state=random_state)

pipeline = Pipeline([
        ('model',  HistGradientBoostingRegressor(max_iter=200, random_state=random_state)),
    ])


# 3 values per param centered around RandomizedSearchCV best:
param_grid = {
    'model__max_iter':      [175, 200, 225],
    'model__learning_rate': [0.02, 0.027, 0.035],
    'model__max_depth':     [6, 7, 8],
}

grid_search = GridSearchCV(pipeline, param_grid, cv=kf,
                           scoring='neg_root_mean_squared_error', n_jobs=-1)

grid_search.fit(X_train_val, y_train_val)

print(f"Best params: {grid_search.best_params_}")
print(f"CV RMSE:     {-grid_search.best_score_:.2f}")

Best params: {'model__learning_rate': 0.027, 'model__max_depth': 8, 'model__max_iter': 200}
CV RMSE:     34.18


Best learning rate stayed the same, but GridSearch picked the highest value for model max depth.  
Max iterations stayed roughly the same

In [6]:

param_grid = {
    'model__max_iter':      [190, 200, 210],
    'model__learning_rate': [0.025, 0.027, 0.03],
    'model__max_depth':     [8,9,10],
}

grid_search = GridSearchCV(pipeline, param_grid, cv=kf,
                           scoring='neg_root_mean_squared_error', n_jobs=-1)

grid_search.fit(X_train_val, y_train_val)

print(f"Best params: {grid_search.best_params_}")
print(f"CV RMSE:     {-grid_search.best_score_:.2f}")

Best params: {'model__learning_rate': 0.027, 'model__max_depth': 8, 'model__max_iter': 210}
CV RMSE:     34.18


In [7]:

param_grid = {
    'model__max_iter':      [200, 210, 220],
    'model__learning_rate': [0.025, 0.027, 0.03],
    'model__max_depth':     [8,9,10],
}

grid_search = GridSearchCV(pipeline, param_grid, cv=kf,
                           scoring='neg_root_mean_squared_error', n_jobs=-1)

grid_search.fit(X_train_val, y_train_val)

print(f"Best params: {grid_search.best_params_}")
print(f"CV RMSE:     {-grid_search.best_score_:.2f}")

Best params: {'model__learning_rate': 0.027, 'model__max_depth': 8, 'model__max_iter': 210}
CV RMSE:     34.18


In [8]:
param_grid = {
    'model__max_iter':      [200, 210, 215],
    'model__learning_rate': [0.027, 0.03, 0.035],
    'model__max_depth':     [8,10,12],
}

grid_search = GridSearchCV(pipeline, param_grid, cv=kf,
                           scoring='neg_root_mean_squared_error', n_jobs=-1)

grid_search.fit(X_train_val, y_train_val)

print(f"Best params: {grid_search.best_params_}")
print(f"CV RMSE:     {-grid_search.best_score_:.2f}")

Best params: {'model__learning_rate': 0.027, 'model__max_depth': 8, 'model__max_iter': 210}
CV RMSE:     34.18


It seems that after doing GridSearchCV there are better parameters than these calculated by RandomizedSearchedCV.
After doing GridSearch model was able to improve on CV by 0.01.

In [9]:
best_model = grid_search.best_estimator_
best_model.fit(X_train_val, y_train_val)

y_pred_test = best_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
print(f"Test RMSE (all features):  {test_rmse:.2f}")

Test RMSE (all features):  34.55


# Final Result

CV RMSE (all features):  34.18 improvement of 0.01 <br>
Test RMSE (all features):  34.55  improvement of 0.02 <br>


Comparing to : <br>
Previous model : <br>
  CV RMSE (all features):  34.19 <br>
Simple Linear Regression:  <br>
  CV   RMSE: 34.98 <br>

Dummy (mean) RMSE: 37.61